# Load: IP charges → Amazon RDS

Validates the mart CSV then writes it to the `country_profiles` schema in Amazon RDS.

| | |
|---|---|
| **Reads** | `data/marts/fct_ip_charges_yearly.csv` |
| **Writes** | `country_profiles.fact_ip_charges_yearly` (PostgreSQL) |

**Prerequisites:**
- Create a `.env` file in the project root with the following variables: `RDS_HOST`, `RDS_PORT`, `RDS_DB`, `RDS_USER`, `RDS_PASSWORD`
- Run the full pipeline first: raw → staging → mart notebooks
- All 15 validation tests must pass before any data is written

**Output schema:**

| Column | Type | Description |
|---|---|---|
| `year` | integer | Reference year |
| `macroecon_region_id` | integer | FK → `dim_macroecon_region.macroecon_region_id` |
| `value_cad_millions` | numeric(20,2) | Net IP charges balance in millions CAD |

In [11]:
import sys
# Install required packages into the active kernel
import subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', 'sqlalchemy', 'psycopg2-binary', 'python-dotenv'], check=True)
print('Packages ready')

Packages ready


In [ ]:
import os
import subprocess
import pandas as pd
import sqlalchemy as sa
from pathlib import Path
from dotenv import load_dotenv

import sys
# Bootstrap: add project root to path so utils can be imported
_here = Path(__file__).parent if '__file__' in dir() else Path.cwd()
# Walk up until we find the project root (contains utils/)
_root = _here
while not (_root / 'utils').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from utils.paths import MARTS_DIR, REPO_DIR

project_root = REPO_DIR
PYTHON = sys.executable
INPUT_FILENAME = 'fct_ip_charges_yearly.csv'

print(f'Project root: {project_root}')

## Step 1 — Validate the CSV

All 15 tests must pass before any data is written to RDS.

In [13]:
result = subprocess.run(
    [PYTHON, '-m', 'pytest', 'tests/test_mart_fct_ip_charges.py', '-v'],
    capture_output=True,
    text=True,
    cwd=str(project_root)
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('Validation failed — fix the CSV before uploading to RDS')
print('All validation tests passed — safe to proceed')

============================= test session starts =============================
platform win32 -- Python 3.11.5, pytest-7.4.0, pluggy-1.0.0 -- C:\ProgramData\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\GuidaK\OneDrive - ISED-ISDE\Documents\Work\or_country_profiles_dashboard
configfile: pytest.ini
plugins: anyio-3.5.0
collecting ... collected 15 items

tests/test_mart_fct_ip_charges.py::test_columns_correct PASSED           [  6%]
tests/test_mart_fct_ip_charges.py::test_row_count PASSED                 [ 13%]
tests/test_mart_fct_ip_charges.py::test_no_nulls PASSED                  [ 20%]
tests/test_mart_fct_ip_charges.py::test_no_empty_strings PASSED          [ 26%]
tests/test_mart_fct_ip_charges.py::test_year_dtype PASSED                [ 33%]
tests/test_mart_fct_ip_charges.py::test_year_range PASSED                [ 40%]
tests/test_mart_fct_ip_charges.py::test_flow_type_single_value PASSED    [ 46%]
tests/test_mart_fct_ip_charges.py::test_value_dtype PASSED         

## Step 2 — Load the CSV

In [14]:
df = pd.read_csv(MARTS_DIR / INPUT_FILENAME)
print(f'Loaded {len(df)} rows from mart CSV')
df.head()

Loaded 1315 rows from mart CSV


,year,counterpart_country,flow_type,value_cad_millions
0,1969,World,Charges for use of IP - Balance,-167
1,1970,World,Charges for use of IP - Balance,0
2,1971,World,Charges for use of IP - Balance,0
3,1972,World,Charges for use of IP - Balance,0
4,1973,World,Charges for use of IP - Balance,-243


## Step 3 — Connect to RDS

Credentials are loaded from `.env` in the project root. Never hardcode them here.

In [15]:
load_dotenv(project_root / '.env')

engine = sa.create_engine(
    sa.engine.URL.create(
        drivername='postgresql+psycopg2',
        username=os.environ['RDS_USER'],
        password=os.environ['RDS_PASSWORD'],
        host=os.environ['RDS_HOST'],
        port=int(os.environ['RDS_PORT']),
        database=os.environ['RDS_DB'],
    )
)

with engine.connect() as conn:
    version = conn.execute(sa.text('SELECT version()')).scalar()
    print(f'Connected: {version[:60]}')

Connected: PostgreSQL 14.22 on aarch64-unknown-linux-gnu, compiled by a


## Step 4 — Load dimension tables from RDS

We need `dim_macroecon_region` to look up integer IDs for each country name.

In [16]:
with engine.connect() as conn:
    dim_region = pd.read_sql(
        sa.text('SELECT macroecon_region_id, region FROM country_profiles.dim_macroecon_region'),
        conn
    )

print(f'Loaded {len(dim_region)} regions from dim_macroecon_region')
dim_region.head(10)

Loaded 272 regions from dim_macroecon_region


,macroecon_region_id,region
0,1,All countries
1,2,Europe
2,3,Belgium
3,4,Bulgaria
4,5,Czechia
5,6,Denmark
6,7,Germany
7,8,Estonia
8,9,Ireland
9,10,Greece


## Step 5 — Map country names to dimension IDs

"World" in the CSV = "All countries" in the database (both represent the same OECD aggregate).
All other names were already standardised in the mart notebook to match `dim_macroecon_region`.

In [17]:
NAME_MAP = {'World': 'All countries'}
df['region_lookup'] = df['counterpart_country'].replace(NAME_MAP)

merged = df.merge(dim_region, left_on='region_lookup', right_on='region', how='left')

# Guard: duplicate rows in dim_region would inflate the merge result
assert len(merged) == len(df), (
    f'Merge produced {len(merged)} rows from {len(df)} — dim_macroecon_region has duplicate region names'
)

unmapped = merged[merged['macroecon_region_id'].isna()]['counterpart_country'].unique()
if len(unmapped) > 0:
    print(f'UNMAPPED COUNTRIES ({len(unmapped)}) — fix before uploading:')
    for c in sorted(unmapped):
        print(f'  - {c}')
    raise ValueError('Unmapped countries found — upload aborted')

print(f'All {df["counterpart_country"].nunique()} countries matched successfully')

All 84 countries matched successfully


## Step 6 — Prepare the final load DataFrame

In [18]:
load_df = (
    merged[['year', 'macroecon_region_id', 'value_cad_millions']]
    .astype({'macroecon_region_id': int, 'value_cad_millions': float})
    .sort_values(['year', 'macroecon_region_id'])
    .reset_index(drop=True)
)

print(f'Final shape: {load_df.shape}')
print(f'Year range: {load_df["year"].min()} – {load_df["year"].max()}')
load_df.head(10)

Final shape: (1315, 3)
Year range: 1969 – 2024


,year,macroecon_region_id,value_cad_millions
0,1969,1,-167.0
1,1970,1,0.0
2,1971,1,0.0
3,1972,1,0.0
4,1973,1,-243.0
5,1974,1,0.0
6,1975,1,0.0
7,1976,1,0.0
8,1977,1,-410.0
9,1978,1,0.0


## Step 7 — Write to RDS

`if_exists='fail'` on first run — errors if the table already exists, forcing a conscious decision.  
Change to `'replace'` only after confirming the table is yours to overwrite.

In [19]:
with engine.begin() as conn:
    load_df.to_sql(
        name='fact_ip_charges_yearly',
        con=conn,
        schema='country_profiles',
        if_exists='fail',
        index=False,
        dtype={
            'year':                sa.Integer(),
            'macroecon_region_id': sa.Integer(),
            'value_cad_millions':  sa.Numeric(20, 2),
        }
    )

print('fact_ip_charges_yearly written to country_profiles schema')

fact_ip_charges_yearly written to country_profiles schema


## Step 8 — Verify

In [20]:
with engine.connect() as conn:
    rds_count = conn.execute(
        sa.text('SELECT COUNT(*) FROM country_profiles.fact_ip_charges_yearly')
    ).scalar()

print(f'Rows in RDS:  {rds_count}')
print(f'Rows in CSV:  {len(load_df)}')

assert rds_count == len(load_df), f'Row count mismatch: expected {len(load_df)}, got {rds_count}'
print('Verification passed — upload complete')

Rows in RDS:  1315
Rows in CSV:  1315
Verification passed — upload complete


## Rollback — run this cell only if you need to undo the upload

Drops the table completely. Safe because this is a new table — nothing else depends on it yet.  
After dropping: fix the issue, re-run the validation tests, then re-run Steps 1–8 above.

In [ ]:
with engine.begin() as conn:
    conn.execute(sa.text('DROP TABLE IF EXISTS country_profiles.fact_ip_charges_yearly'))
print('fact_ip_charges_yearly dropped — ready for a clean re-upload')